# CNN para imágenes

Usa esta plantilla para imágenes.

Ejemplos: MNIST, FashionMNIST, CIFAR-10, clasificación de imágenes pequeñas.

**Trampas principales:**
- PyTorch espera `[batch, canales, alto, ancho]` — no `[batch, alto, ancho, canales]`
- OpenCV carga en BGR, torchvision en RGB — si mezclas, los colores están invertidos
- El `Linear` tras las convoluciones tiene un tamaño que hay que calcular — ver celda de diagnóstico
- `CrossEntropyLoss` ya incluye softmax internamente — NO añadir Softmax al final
- Siempre poner `model.eval()` + `torch.no_grad()` al evaluar

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# SIEMPRE definir device primero — si hay GPU la usa, si no CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Usando device:', device)

## Opción A — Dataset de torchvision (MNIST, CIFAR-10, FashionMNIST)

Torchvision carga en RGB y aplica el transform automáticamente. No hay trampa BGR aquí.

In [ ]:
# --- MNIST (1 canal, 28x28, 10 clases) ---
# Para FashionMNIST: cambia datasets.MNIST por datasets.FashionMNIST, todo lo demás igual

transform_mnist = transforms.Compose([
    transforms.ToTensor(),
    # Normalize(mean, std) — para MNIST estos son los valores estándar
    # Si no los sabes, usar (0.5,) y (0.5,) es siempre aceptable
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform_mnist)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform_mnist)

# shuffle=True SOLO en train — en test no importa el orden
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

# Verificar shape — CRÍTICO antes de construir el modelo
images, labels = next(iter(train_loader))
print('Shape imágenes:', images.shape)   # → [64, 1, 28, 28]  (batch, canales, H, W)
print('Shape etiquetas:', labels.shape)  # → [64]
print('Rango valores:', images.min().item(), 'a', images.max().item())

In [ ]:
# --- CIFAR-10 (3 canales, 32x32, 10 clases) --- CAMBIOS RESPECTO A MNIST:
# 1. in_channels=1 → in_channels=3
# 2. Normalize lleva 3 valores (uno por canal R, G, B)
# 3. El Linear final cambia porque 32x32 → pool → 8x8, no 7x7

transform_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),   # media por canal RGB de CIFAR-10
        std=(0.2470, 0.2435, 0.2616),
    ),
])

# Si no recuerdas los valores exactos de CIFAR, (0.5,0.5,0.5) y (0.5,0.5,0.5) funciona
print('Transform CIFAR definido (no descargar ahora para ahorrar tiempo)')

## Opción B — Dataset propio con carpetas (ImageFolder)

Estructura esperada:
```
data/
  train/
    clase_a/  img1.jpg  img2.jpg ...
    clase_b/  img3.jpg  ...
  test/
    clase_a/  ...
    clase_b/  ...
```

**Trampa BGR/RGB**: si cargas con `cv2.imread()`, las imágenes están en BGR. `PIL` y `ImageFolder` usan RGB. Si mezclas OpenCV con torchvision, los colores están invertidos y el modelo aprende mal.

In [ ]:
from torchvision.datasets import ImageFolder

# Transform para dataset propio (imágenes en color, tamaño variable)
# ResNet y la mayoría de modelos preentrenados esperan 224x224
transform_custom = transforms.Compose([
    transforms.Resize((224, 224)),      # redimensionar a tamaño fijo
    transforms.ToTensor(),              # PIL [H,W,C] uint8 → tensor [C,H,W] float [0,1]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],     # valores de ImageNet — usar siempre con modelos preentrenados
        std=[0.229, 0.224, 0.225],
    ),
])

# Cargar con ImageFolder — detecta clases automáticamente por nombre de carpeta
# train_data = ImageFolder(root='data/train', transform=transform_custom)
# test_data  = ImageFolder(root='data/test',  transform=transform_custom)
# print('Clases detectadas:', train_data.classes)
# print('Muestras train:', len(train_data))

print('ImageFolder listo — descomenta para usar con carpetas reales')

In [ ]:
# TRAMPA BGR/RGB — cómo detectarla y corregirla si cargas con OpenCV
import numpy as np

# Si tienes imágenes cargadas con cv2 (BGR) y quieres convertirlas a tensor:
def bgr_to_tensor(bgr_img):
    """Convierte imagen OpenCV BGR a tensor PyTorch [C, H, W] normalizado."""
    # 1. BGR → RGB
    rgb_img = bgr_img[:, :, ::-1].copy()   # invierte el eje de canales
    # 2. uint8 [0,255] → float32 [0,1]
    rgb_float = rgb_img.astype(np.float32) / 255.0
    # 3. [H, W, C] → [C, H, W]
    tensor = torch.tensor(rgb_float).permute(2, 0, 1)
    return tensor

# Verificar que la conversión es correcta:
# img_cv2 = cv2.imread('foto.jpg')   # BGR
# t = bgr_to_tensor(img_cv2)
# print(t.shape)  # → [3, H, W]
print('Función bgr_to_tensor definida')

## Data Augmentation

Solo se aplica en **train**, nunca en test/val.
El objetivo es que el modelo vea variaciones artificiales y generalice mejor.

In [ ]:
# Augmentation en train — test usa transform sin augmentation
transform_train_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),       # voltear horizontal (no usar si la orientación importa)
    transforms.RandomRotation(degrees=10),         # rotar ±10 grados
    transforms.ColorJitter(brightness=0.2,         # variaciones de brillo/contraste
                           contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

transform_test_no_aug = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

# NUNCA apliques RandomHorizontalFlip en test — los resultados serían distintos cada vez
print('Transforms con augmentation definidos')

## Arquitectura CNN

**Fórmula tamaño de salida de Conv2d:**
```
output = floor((input + 2*padding - kernel_size) / stride) + 1
```
Con `padding=1, kernel_size=3, stride=1` → output = input (no cambia tamaño)  
Con `MaxPool2d(2)` → output = input / 2

**Para MNIST 28x28:**  
Conv1(pad=1) → 28x28 → Pool(2) → 14x14  
Conv2(pad=1) → 14x14 → Pool(2) → 7x7  
Flatten → 64 * 7 * 7 = **3136**

**Para CIFAR 32x32:**  
Conv1(pad=1) → 32x32 → Pool(2) → 16x16  
Conv2(pad=1) → 16x16 → Pool(2) → 8x8  
Flatten → 64 * 8 * 8 = **4096**

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, in_channels: int = 1, num_classes: int = 10) -> None:
        super().__init__()
        # Bloque 1: [B, in_channels, H, W] → [B, 32, H/2, W/2]
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),  # mismo tamaño
            nn.ReLU(),
            nn.MaxPool2d(2),    # divide dimensiones espaciales por 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),    # divide por 2 otra vez
        )
        # Bloque clasificador: depende del tamaño de entrada
        # MNIST: 64*7*7=3136 | CIFAR/FashionMNIST: 64*8*8=4096 | 64x64: 64*16*16=16384
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),  # ← CAMBIAR si input no es 28x28
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
            # NO añadir Softmax — CrossEntropyLoss lo incluye
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)


model = SimpleCNN(in_channels=1, num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros entrenables: {total_params:,}')

In [ ]:
# DIAGNÓSTICO DE TAMAÑO — si no sabes el tamaño correcto del Linear, usa esto
# Pasa un batch falso por self.features y mira el shape resultante

with torch.no_grad():
    dummy = torch.zeros(1, 1, 28, 28).to(device)  # batch=1, canales=1, H=28, W=28
    out = model.features(dummy)
    print('Shape tras features:', out.shape)       # → [1, 64, 7, 7]
    print('Linear input size:', out.view(1, -1).shape[1])  # → 3136
    # Ese número va en nn.Linear(AQUÍ, 128)

## Entrenamiento con validación

Siempre monitorizar **train loss** y **val/test accuracy** por epoch para detectar overfitting a tiempo.

In [ ]:
import matplotlib.pyplot as plt

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()   # activa Dropout y BatchNorm en modo entrenamiento
    total_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()           # limpiar gradientes del paso anterior
        logits = model(images)          # forward
        loss   = criterion(logits, labels)
        loss.backward()                 # backprop
        optimizer.step()                # actualizar pesos
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, device):
    model.eval()    # desactiva Dropout y fija BatchNorm
    correct = total = 0
    with torch.no_grad():   # no calcular gradientes → más rápido y menos memoria
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total


EPOCHS = 5
history = {'train_loss': [], 'test_acc': []}

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_acc   = evaluate(model, test_loader, device)

    history['train_loss'].append(train_loss)
    history['test_acc'].append(test_acc)

    print(f'Epoch {epoch}/{EPOCHS}  loss={train_loss:.4f}  test_acc={test_acc:.4f}')

In [ ]:
# Graficar curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].plot(history['train_loss'])
axes[0].set_title('Train Loss')
axes[0].set_xlabel('Epoch')

axes[1].plot(history['test_acc'])
axes[1].set_title('Test Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print(f'Accuracy final: {history["test_acc"][-1]:.4f}')

## Transfer Learning — usar ResNet preentrenada

Cuándo usarlo: pocas imágenes propias, dataset pequeño, clases similares a ImageNet.

**Estrategia 1 (examen por defecto):** congela todo, entrena solo la última capa.  
**Estrategia 2 (fine-tuning):** descongela las últimas capas con lr bajo.

In [ ]:
from torchvision import models

# Cargar ResNet18 preentrenada — 'pretrained=True' está deprecado, usar weights=
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Estrategia 1: Feature extraction
# Congelar TODOS los parámetros
for param in resnet.parameters():
    param.requires_grad = False

# Reemplazar la capa final (fc) por una nueva con tus clases
NUM_CLASSES = 10  # CAMBIAR
resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
# resnet.fc es nueva → requires_grad=True por defecto

# Solo el optimizador apunta a la capa nueva
optimizer_tl = torch.optim.Adam(resnet.fc.parameters(), lr=0.001)

resnet = resnet.to(device)

trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total     = sum(p.numel() for p in resnet.parameters())
print(f'Entrenables: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)')

In [ ]:
# Estrategia 2: Fine-tuning de las últimas capas
resnet2 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Congelar todo primero
for param in resnet2.parameters():
    param.requires_grad = False

# Descongelar el último bloque (layer4) + capa final
for param in resnet2.layer4.parameters():
    param.requires_grad = True

resnet2.fc = nn.Linear(resnet2.fc.in_features, NUM_CLASSES)

# Learning rates distintos: bajo para capas preentrenadas, alto para la capa nueva
optimizer_ft = torch.optim.Adam([
    {'params': resnet2.layer4.parameters(), 'lr': 1e-4},
    {'params': resnet2.fc.parameters(),     'lr': 1e-3},
])

print('Fine-tuning: layer4 + fc entrenables')

# El loop de entrenamiento es el mismo que SimpleCNN
# IMPORTANTE: el transform debe incluir Resize(224) y los valores de ImageNet:
# transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

## Resumen de cambios según el dataset

| Dataset | in_channels | Normalize | Linear in | Transform extra |
|---|---|---|---|---|
| MNIST / FashionMNIST | 1 | `(0.5,),(0.5,)` | 64\*7\*7=3136 | — |
| CIFAR-10 | 3 | `(0.5,0.5,0.5)` | 64\*8\*8=4096 | — |
| Dataset propio ~64px | 3 | `(0.5,0.5,0.5)` | 64\*16\*16=16384 | `Resize(64)` |
| ResNet preentrenada | 3 | ImageNet stats | — (fc automático) | `Resize(224)` |

## Si falla

- **Error en Linear**: el tamaño 64\*7\*7 no coincide → usa la celda de diagnóstico (dummy forward)
- **Loss no baja**: ¿aplicaste Normalize? ¿el lr es muy alto/bajo? Prueba `lr=0.001` (Adam por defecto)
- **Accuracy estancada en 1/num_clases**: el modelo adivina al azar → revisa etiquetas y shapes
- **Imágenes se ven raras**: posiblemente BGR/RGB invertido — usa `img[:,:,::-1]` o `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`
- **CUDA out of memory**: baja `batch_size` a 32 o 16
- **pretrained=True deprecation warning**: cambia a `weights=models.ResNet18_Weights.IMAGENET1K_V1`